# GLiNER fine-tune — Bosch GDPR PII (Colab)

Runs on a free Colab T4 in ~30-60min for 3 epochs over ~3.7K examples.

**Steps**
1. Set runtime to GPU (Runtime → Change runtime type → T4 GPU)
2. Run all cells top-to-bottom
3. Trained model lands in `/content/drive/MyDrive/gliner-bosch/` (if Drive mounted)

In [ ]:
# 1. GPU check
!nvidia-smi | head -20

In [ ]:
# 2. Clone (or pull) repo
import os
REPO_URL = 'https://github.com/mohanadkandil/tracer.git'  # public? if private use token URL
REPO_DIR = '/content/repo'
if os.path.exists(REPO_DIR):
    !cd $REPO_DIR && git pull
else:
    !git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!ls train/ && echo "--- data ---" && ls data/ 2>/dev/null || echo "data/ missing — run cell 4"

In [ ]:
# 3. Install deps + register repo as editable package so `python -m train.finetune` resolves
!pip install -q gliner>=0.2.13 transformers>=4.40.0 accelerate>=0.30.0 seqeval>=1.2.2 orjson faker jinja2 pydantic httpx python-dotenv tqdm
%cd /content/repo
!pip install -q -e .
import sys; sys.path.insert(0, '/content/repo')
!ls train/ data_gen/

In [ ]:
# 4. Convert data → GLiNER token format (run after data/out/*.jsonl exists in repo)
import os
REPO_DIR = '/content/repo'
os.environ['PYTHONPATH'] = REPO_DIR
!cd $REPO_DIR && PYTHONPATH=$REPO_DIR python -m train.convert --in-dir data/out --out-dir data/gliner
!ls -lh $REPO_DIR/data/gliner/

In [ ]:
# 5. Mount Drive so the model checkpoint survives session disconnect
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/gliner-bosch'
!mkdir -p $OUT_DIR

In [ ]:
# 6. Train — bulletproof: absolute PYTHONPATH + explicit cd + verify before launch
import os
REPO_DIR = '/content/repo'
assert os.path.isfile(f'{REPO_DIR}/train/finetune.py'), 'train/finetune.py missing — re-run cell 2'
assert os.path.isfile(f'{REPO_DIR}/data/gliner/train.jsonl'), 'data/gliner/train.jsonl missing — re-run cell 4'
os.environ['PYTHONPATH'] = REPO_DIR
os.chdir(REPO_DIR)
print('cwd:', os.getcwd(), '\nPYTHONPATH:', os.environ['PYTHONPATH'])

!PYTHONPATH=$REPO_DIR python -m train.finetune \
    --train data/gliner/train.jsonl \
    --val data/gliner/val.jsonl \
    --base-model urchade/gliner_multi_pii-v1 \
    --out $OUT_DIR \
    --epochs 3 \
    --batch-size 16 \
    --lr 5e-6 \
    --lr-others 1e-5

In [ ]:
# 7. Sanity-check the trained model on a held-out example
from gliner import GLiNER
model = GLiNER.from_pretrained(OUT_DIR, local_files_only=True)
labels = ['PERSON', 'EMPLOYEE_ID', 'EMAIL', 'PHONE', 'ADDRESS', 'TAX_ID', 'IBAN', 'DEPARTMENT', 'COMPANY', 'DATE', 'SIGNATURE', 'USERNAME']
sample = (
    'Spesenabrechnung (ausgefüllt)\n'
    'Mitarbeiter: Hans Müller (E-43217)\n'
    'Abteilung: Forschung & Entwicklung\n'
    'Datum: 12 Mär 2026\n'
    'Vorgesetzter: Anna Becker\n'
    'Unterschrift: A. Becker'
)
for ent in model.predict_entities(sample, labels, threshold=0.4):
    print(f"{ent['label']:<14} {ent['text']!r}  (score={ent['score']:.2f})")

In [ ]:
# 8. Zip + download checkpoint for use in scan service
import shutil
shutil.make_archive('/content/gliner-bosch', 'zip', OUT_DIR)
from google.colab import files
files.download('/content/gliner-bosch.zip')